# 02 — Nelson-Siegel Decomposition (5 Countries)

Fit Level, Slope, Curvature factors for UK, US, EU, CA, BR.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.nelson_siegel import (
    ns_factor_loadings, fit_ns_ols, compute_rmse, select_lambda,
)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)

MATURITIES = {
    'US': np.array([1, 2, 3, 5, 10, 20], dtype=float),
    'EU': np.array([1, 2, 3, 5, 10, 20], dtype=float),
    'CA': np.array([2, 3, 5, 7, 10, 30], dtype=float),
    'BR': np.array([1, 3, 6, 12], dtype=float),
    'UK': np.array([1, 2, 3, 5, 10, 20], dtype=float),
}

ALL_COUNTRIES = ['US', 'EU', 'CA', 'BR', 'UK']

## 1. Load Data

In [ ]:
combined = pd.read_csv('../data/raw/aligned_yields_5c.csv', index_col=0, parse_dates=True)
print(f'Shape: {combined.shape}')

## 2. Fit NS Factors Per Country

In [ ]:
all_factors = {}
lambdas = {}

for cc in ALL_COUNTRIES:
    cols = [c for c in combined.columns if c.startswith(f'{cc}_')]
    yields = combined[cols]
    mats = MATURITIES[cc]

    lam_result = select_lambda(yields, mats)
    best_lam = lam_result['best_lambda']
    lambdas[cc] = best_lam

    factors = fit_ns_ols(yields, mats, lam=best_lam)
    rmse = compute_rmse(yields, factors, mats, lam=best_lam)

    print(f'{cc}: lambda={best_lam:.4f}, RMSE={rmse.mean():.4f} bps, shape={factors.shape}')
    all_factors[cc] = factors

## 3. Factor Time Series

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
for i, factor in enumerate(['Level', 'Slope', 'Curvature']):
    ax = axes[i]
    for cc in ALL_COUNTRIES:
        ax.plot(all_factors[cc].index, all_factors[cc][factor],
                label=cc, alpha=0.8, lw=0.7)
    ax.set_ylabel(factor)
    ax.legend(loc='upper right', fontsize=7)
plt.suptitle('Nelson-Siegel Factors (5 Countries)', fontsize=12)
plt.tight_layout()
plt.show()

## 4. Save

In [ ]:
factor_dfs = [all_factors[cc].add_prefix(f'{cc}_') for cc in ALL_COUNTRIES]
all_df = pd.concat(factor_dfs, axis=1)
all_df.to_csv('../data/factors/ns_factors_5c.csv')
print(f'Saved ns_factors_5c.csv: {all_df.shape}')
all_df.describe().round(2)